# 04 - Non-IID: saat klien tidak seragam (TANPA SFM)

**Tujuan belajar:** melihat FedAvg menurun ketika distribusi label antar-klien berbeda
(non-IID), dan mengukur seberapa tajam penurunannya lewat parameter Dirichlet alpha.

- **alpha besar (mis. 100)** -> mendekati IID (tiap klien mirip populasi).
- **alpha kecil (mis. 0.1)** -> sangat non-IID (klien didominasi satu kelas).

Kita partisi ULANG data train global memakai Dirichlet pada label, lalu jalankan
FedAvg yang sama seperti nb03 untuk beberapa nilai alpha. Ini versi belajar dari
`02_survey-tanpaSFM/s05_noniid_sweep.ipynb`.

**Prasyarat:** `01_data_partition.ipynb` (kita pakai `fl_data/global_test.npz` dan
gabungan train klien sebagai populasi untuk dipartisi ulang). Dijalankan di SageMaker.


In [ ]:
import sys
!{sys.executable} -m pip install -q numpy scikit-learn
print('deps OK')

## 1. MLP + FedAvg (sama seperti nb03)


In [ ]:
import numpy as np
SEED=42
def init_weights(d, seed=42):
    r=np.random.RandomState(seed); L=[d,128,64,1]; W=[]
    for a,b in zip(L[:-1],L[1:]):
        W.append(r.randn(a,b)*np.sqrt(2.0/a)); W.append(np.zeros(b))
    return W
def _relu(x): return np.maximum(0,x)
def _sig(x):  return 1.0/(1.0+np.exp(-np.clip(x,-30,30)))
def forward(W,X):
    a1=X@W[0]+W[1]; z1=_relu(a1)
    a2=z1@W[2]+W[3]; z2=_relu(a2)
    a3=z2@W[4]+W[5]; return _sig(a3).ravel(),(X,a1,z1,a2,z2,a3,None)
def train_local(W,X,y,epochs=2,lr=3e-3,batch=256,mu=0.0,W_global=None,seed=42):
    r=np.random.RandomState(seed); n=len(y); W=[w.copy() for w in W]
    for _ in range(epochs):
        idx=r.permutation(n)
        for s in range(0,n,batch):
            bi=idx[s:s+batch]; Xb=X[bi]; yb=y[bi].astype(float)
            o,(X_,a1,z1,a2,z2,a3,_)=forward(W,Xb)
            g=(o-yb)/len(yb)
            gW4=z2.T@g[:,None]; gb4=g.sum(0)
            d2=(g[:,None]@W[4].T)*(a2>0); gW2=z1.T@d2; gb2=d2.sum(0)
            d1=(d2@W[2].T)*(a1>0);        gW0=X_.T@d1; gb0=d1.sum(0)
            for i,gv in enumerate([gW0,gb0,gW2,gb2,gW4,gb4]):
                gv=np.asarray(gv).reshape(W[i].shape)
                if mu>0 and W_global is not None: gv=gv+mu*(W[i]-W_global[i])
                W[i]=W[i]-lr*gv
    return W
def fedavg(ws, sizes):
    tot=float(sum(sizes)); return [sum((sizes[k]/tot)*ws[k][i] for k in range(len(ws))) for i in range(len(ws[0]))]
def predict(W,X): return forward(W,X)[0]
def zfit(X): mu=X.mean(0); sd=X.std(0); sd[sd==0]=1.0; return mu,sd
def zapp(X,mu,sd): return (X-mu)/sd
print('ok')

## 2. Muat populasi train + test global


In [ ]:
import os
DATA='./fl_data'
assert os.path.isdir(DATA), 'Jalankan 01_data_partition.ipynb dulu'
cs=sorted(set(int(f.split('_')[1]) for f in os.listdir(DATA) if f.startswith('client_')))
Xpop=np.vstack([np.load(f'{DATA}/client_{i}_train.npz')['X'] for i in cs])
ypop=np.concatenate([np.load(f'{DATA}/client_{i}_train.npz')['y'] for i in cs])
g=np.load(f'{DATA}/global_test.npz'); Xg,yg=g['X'],g['y']; d=Xg.shape[1]
K=4
print('populasi train:', Xpop.shape, '| attack%:', round(100*ypop.mean(),2), '| d =', d)

## 3. Partisi Dirichlet pada label

Untuk tiap kelas, bagi indeksnya ke K klien menurut proporsi ~ Dirichlet(alpha).
alpha kecil membuat satu klien memborong mayoritas suatu kelas (non-IID ekstrem).


In [ ]:
def dirichlet_partition(y, K, alpha, seed=42):
    r=np.random.RandomState(seed); idx_by_k=[[] for _ in range(K)]
    for c in np.unique(y):
        ci=np.where(y==c)[0]; r.shuffle(ci)
        prop=r.dirichlet([alpha]*K)
        cuts=(np.cumsum(prop)*len(ci)).astype(int)[:-1]
        for k,part in enumerate(np.split(ci,cuts)): idx_by_k[k]+=part.tolist()
    return [np.array(sorted(ix)) for ix in idx_by_k]

# contoh: tunjukkan komposisi untuk alpha kecil vs besar
for a in (0.1, 100.0):
    parts=dirichlet_partition(ypop, K, a, SEED)
    comp=[round(100*ypop[ix].mean(),1) for ix in parts]
    print(f'alpha={a:>5}: attack% per klien = {comp}')

## 4. Sweep alpha: jalankan FedAvg untuk tiap tingkat heterogenitas


In [ ]:
from sklearn.metrics import matthews_corrcoef
def run_fedavg(parts, rounds=30, local_epochs=2, mu=0.0):
    cli=[]
    for ix in parts:
        Xi,yi=Xpop[ix],ypop[ix]; mu_i,sd_i=zfit(Xi)
        cli.append(dict(Xz=zapp(Xi,mu_i,sd_i), y=yi, mu=mu_i, sd=sd_i, n=len(yi)))
    W=init_weights(d,SEED)
    for t in range(rounds):
        cw=[]; sz=[]
        for c in cli:
            if c['n']<10: continue
            cw.append(train_local(W,c['Xz'],c['y'],epochs=local_epochs,mu=mu,W_global=W,seed=SEED)); sz.append(c['n'])
        W=fedavg(cw,sz)
    ms=[matthews_corrcoef(yg,(predict(W,zapp(Xg,c['mu'],c['sd']))>=0.5).astype(int)) for c in cli]
    return float(np.mean(ms))

ALPHAS=[100.0, 1.0, 0.5, 0.1]
res=[]
for a in ALPHAS:
    parts=dirichlet_partition(ypop,K,a,SEED)
    m=run_fedavg(parts); res.append((a,m))
    print(f'alpha={a:>5}: FedAvg MCC global = {m:.4f}')

## 5. Simpan + titik putus

Simpan kurva alpha vs MCC. 'Titik putus' = alpha di mana performa anjlok tajam.


In [ ]:
import csv
os.makedirs('learn_out',exist_ok=True)
with open('learn_out/04_noniid.csv','w',newline='') as f:
    w=csv.writer(f); w.writerow(['alpha','mcc_global']); [w.writerow([a,round(m,4)]) for a,m in res]
best=res[0][1]; worst=res[-1][1]
print('tersimpan: learn_out/04_noniid.csv')
print(f'IID-ish (alpha=100) MCC={best:.4f}  ->  non-IID (alpha=0.1) MCC={worst:.4f}  (turun {best-worst:+.4f})')
print('Langkah berikut: 05_fedprox.ipynb -- meredam penurunan non-IID dengan proximal term.')